In [1]:
import accelforge as af
from scheduling.scheduler import *
from af_wrapper import *
import numpy as np


def run(
    einsums,
    compute_units,
    data_dependencies,
    mapping_grid = None,
    latency_grid = None,
    memory_name = None
):
    schedule, min_latency = best_schedule(
        einsums,
        compute_units,
        data_dependencies,
        mapping_grid,
        latency_grid,
        memory_name
    )
    print(schedule)
    print(min_latency)
    return schedule, min_latency


In [15]:
# Toy example. This only works with the non-resource-aware scheduler,
# which takes a grid of latencies as inputs rather than mappings
data_dependencies = {
    "e1": [],
    "e2": [],
    "e3": ["e1", "e2"]
}
compute_units = ['slow', 'fast']
einsums = data_dependencies.keys()
grid = {
    ('slow', 'e1'): 0.2,
    ('fast', 'e1'): 0.1,
    ('slow', 'e2'): 0.1,
    ('fast', 'e2'): 0.05,
    ('slow', 'e3'): 0.2,
    ('fast', 'e3'): 0.1,
}
run(einsums, compute_units, data_dependencies, latency_grid=grid)


{(e1, fast, latency=0.1): 0, (e2, slow, latency=0.1): 0, (e3, fast, latency=0.1): 0.1}
0.2


({(e1, fast, latency=0.1): 0,
  (e2, slow, latency=0.1): 0,
  (e3, fast, latency=0.1): 0.1},
 0.2)

In [3]:
arch = "arch/eyeriss-submods/full.yaml"
m2_workload = "workload/milestone-2/m2-full.yaml"

In [4]:
spec = af.Spec.from_yaml(
    arch,
    m2_workload
)

In [5]:
# m2_full_mapping = af_map(
#     "arch/eyeriss-submods/full.yaml",
#     "workload/milestone-2/m2-full.yaml"
# )

In [6]:
# mappings = []
# for i in range(5):
#     filename = trunc(arch) + "-" + trunc(m2_workload) + "-" + str(i)
#     with open("images/" + filename + ".svg", "w") as f:
#         f.write(m2_full_mapping[i].render())
#     mappings.append(process_mapping(arch, m2_workload, m2_full_mapping[i]))

# # m2_full_mapping.to_dict()
# mappings

In [7]:
data_dependencies = {
    "E0": [],
    "E1": [],
    "E2": ["E0"],
    "E3": ["E0", "E1"], # weirdly we get a cycle if we have a different dep graph where E3 not dep on E0
    "E4": ["E2", "E3"]
}
compute_units = ['slow', 'fast']
einsums = data_dependencies.keys()
memory_name = "MainMemory"

In [8]:
%%capture grid_output
# The code and the concrete values generated,
# to skip recomputation when running with fresh kernel

grid = af_grid(
    einsums, 
    compute_units,
    lambda einsum: "workload/milestone-2/"+einsum+".yaml",
    lambda sub_arch: "arch/eyeriss-submods/"+sub_arch+".yaml"
)

# grid_lats = {('slow', 'E0'): np.float32(0.00049152),
#  ('slow', 'E1'): np.float32(2.08e-05),
#  ('slow', 'E2'): np.float32(0.00049152),
#  ('slow', 'E3'): np.float32(2.08e-05),
#  ('slow', 'E4'): np.float32(2.08e-05),
#  ('fast', 'E0'): np.float32(0.00018412955),
#  ('fast', 'E1'): np.float32(2.08e-05),
#  ('fast', 'E2'): np.float32(0.00018412955),
#  ('fast', 'E3'): np.float32(2.08e-05),
#  ('fast', 'E4'): np.float32(2.08e-05)}

In [9]:
(grid_lats, grid_mems, grid_maps) = grid

In [10]:
grid_lats

{('slow', 'E0'): np.float32(0.00049152),
 ('slow', 'E1'): np.float32(2.08e-05),
 ('slow', 'E2'): np.float32(0.00049152),
 ('slow', 'E3'): np.float32(2.08e-05),
 ('slow', 'E4'): np.float32(2.08e-05),
 ('fast', 'E0'): np.float32(0.00018412955),
 ('fast', 'E1'): np.float32(2.08e-05),
 ('fast', 'E2'): np.float32(0.00018412955),
 ('fast', 'E3'): np.float32(2.08e-05),
 ('fast', 'E4'): np.float32(2.08e-05)}

In [11]:
# for (key, mapping) in grid_maps.items():
#     print(mapping.latency(per_component=True))
#     print("Overall latency:", mapping.latency(per_einsum=True))
#     print()
#     print()

In [12]:
grid_maps[('fast', 'E0')].actions(per_component=True)

{('FastInputScratchpad', 'read'): np.float64(16777216.0),
 ('FastInputScratchpad', 'write'): np.float32(131072.0),
 ('MainMemory', 'read'): np.float64(262144.0),
 ('MainMemory', 'write'): np.float64(131072.0),
 ('FastWeightScratchpad', 'read'): np.float64(16777216.0),
 ('FastWeightScratchpad', 'write'): np.float32(1.6777216e+07),
 ('FastGlobalBuffer', 'read'): np.float32(32768.0),
 ('FastGlobalBuffer', 'write'): np.float32(2048.0),
 ('FastOutputScratchpad', 'read'): np.float32(1.6777216e+07),
 ('FastOutputScratchpad', 'write'): np.float32(1.6777216e+07),
 ('FastMAC', 'compute'): np.float64(2097152.0)}

In [13]:
schedule, latency = run(einsums, compute_units, data_dependencies, grid_maps, grid_lats, memory_name)

(E1, slow, latency=2.080000012938399e-05)
{}
[]
133120.0
1.0
1
0
2.080000012938399e-05
2.080000012938399e-05


(E0, slow, latency=0.0004915199824608862)
{(E1, slow, latency=2.080000012938399e-05): 0}
[{'einsum': 'E1', 'start': 0, 'end': np.float64(2.080000012938399e-05), 'bwu': np.float32(1.0)}]
393216.0
0.125
1
2.080000012938399e-05
0.0005123199825902702
0.0004915199824608862


(E3, slow, latency=2.080000012938399e-05)
{(E1, slow, latency=2.080000012938399e-05): 0, (E0, slow, latency=0.0004915199824608862): np.float64(2.080000012938399e-05)}
[{'einsum': 'E1', 'start': 0, 'end': np.float64(2.080000012938399e-05), 'bwu': np.float32(1.0)}, {'einsum': 'E0', 'start': np.float64(2.080000012938399e-05), 'end': np.float64(0.0005123199825902702), 'bwu': np.float32(0.125)}]
133120.0
1.0
1
0.0005123199825902702
0.0005331199827196542
2.080000012938399e-05


(E2, slow, latency=0.0004915199824608862)
{(E1, slow, latency=2.080000012938399e-05): 0, (E0, slow, latency=0.0004915199824608862): np.float6

In [14]:
[(node, node.latency) for node in schedule.keys()]

[((E0, fast, latency=0.0001841295452322811),
  np.float64(0.0001841295452322811)),
 ((E2, fast, latency=0.0001841295452322811),
  np.float64(0.0001841295452322811)),
 ((E1, slow, latency=3.121614456520912e-05),
  np.float64(3.121614456520912e-05)),
 ((E3, slow, latency=3.121614456520912e-05),
  np.float64(3.121614456520912e-05)),
 ((E4, slow, latency=2.080000012938399e-05),
  np.float64(2.080000012938399e-05))]

In [ ]:
mems = sum((grid_mems[(node.compute_unit, node.einsum_name)] for node in schedule.keys()))

In [ ]:
schedule

In [ ]:
mems

In [20]:
mems # old schedule: np.float64(1185792.0)

np.float64(1185792.0)

In [19]:
(mems/latency) / 6.4e9 # old schedule: np.float64(0.4762258443913986)

np.float64(0.4762258443913986)